In [2]:
import os
import pandas as pd
import numpy as np
import sys
import json
from pathlib import Path

import pyvista as pv 
import pyvista
from pyvista import demos
from pyvista import examples 
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# inserting the lib folder to the compiler
sys.path.insert(0, './lib')
sys.path.insert(0, './utils/')

import utils_misc

from lib_data import DATA_IO
import lib_Subcortical_Atlases

C:\Users\a.kaymak\Anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [3]:
PATH_CURR           = os.path.abspath(os.curdir)    # current code
PATH                = (str(Path(PATH_CURR).parent)) # data repository: upper directory where datasets situated
SUB_LIST            = utils_misc.get_SUB_list(DATA_IO.path_data) # get the SUB id list which we have a recording of them


basal_ganglia       = lib_Subcortical_Atlases.Distal_Atlas(threshold=0.5)
contact_coordinates = pd.read_csv(DATA_IO.path_coordinates + "contact_coordinates.csv")
event_files         = utils_misc.get_files_with_specific_format(DATA_IO.path_events, suffix=".pkl")

# 1. CREATE THE CHANNEL PARCELLATION & MAP DATAFRAME

In [ ]:
df_channel_stn_position = pd.DataFrame(columns=["patient", "hemisphere", "channel", "x", "y", "z", 
                                                "contact1_area", "contact2_area", "channel_area"])

for file in event_files:
    if("LFP" in file):
        
        SUB           = file[4:7]
        df_lfp_events = pd.read_pickle(DATA_IO.path_events + file)
        LFP_channels  = df_lfp_events.LFP_channel.unique()

        print("... SUB-" + SUB)
        
        for HEMISPHERE in ["right", "left"]:
            for CHANNEL in LFP_channels:
                # get the contact names
                CONTACT1 = int(CHANNEL.split("-")[0])
                CONTACT2 = int(CHANNEL.split("-")[1])

                # get the contact coordinates
                contact_1_coor = contact_coordinates[(contact_coordinates.patient == int(SUB)) & 
                                                     (contact_coordinates.hemisphere == HEMISPHERE) & 
                                                     (contact_coordinates.recording_type == "lfp") &  
                                                     (contact_coordinates.contact == CONTACT1)]
                contact_2_coor = contact_coordinates[(contact_coordinates.patient == int(SUB)) & 
                                                     (contact_coordinates.hemisphere == HEMISPHERE) & 
                                                     (contact_coordinates.recording_type == "lfp") &  
                                                     (contact_coordinates.contact == CONTACT2)]
                row               = {}
                row["patient"]    = SUB
                row["hemisphere"] = HEMISPHERE
                row["channel"]    = CHANNEL
    
                if((len(contact_1_coor) != 0) & (len(contact_2_coor) != 0)):

                    # get the coordinates of the contacts
                    try:
                        row["contact1_area"]  = basal_ganglia.check_position_in_nucleus(basal_ganglia.STN, HEMISPHERE,
                                                                                        [[float(contact_1_coor.x), float(contact_1_coor.y), float(contact_1_coor.z)]])
                        row["contact2_area"]  = basal_ganglia.check_position_in_nucleus(basal_ganglia.STN, HEMISPHERE,
                                                                                        [[float(contact_2_coor.x), float(contact_2_coor.y), float(contact_2_coor.z)]])
                    except:
                        print(contact_1_coor)
                        print(contact_2_coor)
                    row["x"]              = (float(contact_1_coor.x) + float(contact_2_coor.x)) / 2
                    row["y"]              = (float(contact_1_coor.y) + float(contact_2_coor.y)) / 2
                    row["z"]              = (float(contact_1_coor.z) + float(contact_2_coor.z)) / 2

                    # check the position of channels (correspond to functional area within STN)
                    row["channel_area"]   = basal_ganglia.check_position_in_nucleus(basal_ganglia.STN, HEMISPHERE,
                                                                                   [[row["x"], row["y"], row["z"]]])         

                    # In same case, one of the two contacts resides outside of  STN meanwhile the other one situated within the STN.
                    # For this particular case, if also estimated position of the channel resides outside of the STN, we assign channel area same 
                    # with the contact that situated in STN.
 
                    if(((row["contact1_area"])=="outside") & ((row["contact2_area"])!="outside") & ((row["channel_area"])=="outside")):
                        row["channel_area"] = row["contact2_area"]
                        
                    elif(((row["contact1_area"])!="outside") & ((row["contact2_area"])=="outside") & ((row["channel_area"])=="outside")):
                        row["channel_area"] = row["contact1_area"]
                        
                    df_channel_stn_position.loc[len(df_channel_stn_position)] = row 

df_channel_stn_position.to_pickle(DATA_IO.path_coordinates + "lfp_channel_stn_parcellation.pkl")

# 2. DIVIDE LFP EVENTS ACROSS PATIENTS BASED ON CHANNEL STN LOCATION

In [7]:
df_channel_stn_position   = pd.read_pickle(DATA_IO.path_coordinates + "lfp_channel_stn_parcellation.pkl")

# create empty dataframe with same column structure of event dataframes

# tapping
df_lfp_tapping_events_motor_area        = pd.DataFrame(columns = ['patient', 'event_no', 'event_category', 'event_laterality',
                                                                  'event_start_time', 'duration', 'LFP_hemisphere', 'LFP_channel',
                                                                  'pre_event_recording', 'event_recording', 'post_event_recording',
                                                                  'CDRS_right_hand', 'CDRS_left_hand', 'CDRS_total', 'dyskinesia_arm','dyskinesia_total'])


df_lfp_tapping_events_associative_area  = pd.DataFrame(columns=df_lfp_tapping_events_motor_area.columns)
df_lfp_tapping_events_limbic_area       = pd.DataFrame(columns=df_lfp_tapping_events_motor_area.columns)

In [9]:
for file in event_files:
    
    if("LFP" in file):
        
        df_lfp_events = pd.read_pickle(DATA_IO.path_events + file)
        SUB           = df_lfp_events.iloc[0].patient
        print("... SUB-" + SUB)

        for index, event_row in df_lfp_events.iterrows():
    
            event_category = event_row.event_category
            hemisphere     = event_row.LFP_hemisphere
            channel        = event_row.LFP_channel
        
            stn_area_row   = df_channel_stn_position[(df_channel_stn_position.patient == SUB) & 
                                                     (df_channel_stn_position.hemisphere == hemisphere) &
                                                     (df_channel_stn_position.channel == channel)]
            
            if(len(stn_area_row)!=0): # if the location of the specified channel is known
                
                stn_area =  stn_area_row.iloc[0,8]
                
                if(stn_area=="motor"):
                    df_lfp_tapping_events_motor_area.loc[len(df_lfp_tapping_events_motor_area)] = event_row
                elif(stn_area=="associative"):
                    df_lfp_tapping_events_associative_area.loc[len(df_lfp_tapping_events_associative_area)] = event_row
                elif(stn_area=="limbic"):
                    df_lfp_tapping_events_limbic_area.loc[len(df_lfp_tapping_events_limbic_area)] = event_row

... SUB-008
... SUB-009
... SUB-010
... SUB-012
... SUB-013
... SUB-014
... SUB-016
... SUB-017
... SUB-019
... SUB-020
... SUB-021
... SUB-022
... SUB-023
... SUB-101
... SUB-102
... SUB-103
... SUB-105
... SUB-107
... SUB-108
... SUB-109
... SUB-110


In [10]:
df_lfp_tapping_events_motor_area.to_pickle(DATA_IO.path_events + "LFP_TAPPING_EVENTS_MOTOR_AREA.pkl")
df_lfp_tapping_events_associative_area.to_pickle(DATA_IO.path_events + "LFP_TAPPING_EVENTS_ASSOCIATIVE_AREA.pkl")
df_lfp_tapping_events_limbic_area.to_pickle(DATA_IO.path_events + "LFP_TAPPING_EVENTS_LIMBIC_AREA.pkl")